# 🎬 ALTO MONTE · Render de video desde el guion escena por escena

Convierte el **JSON** que exporta tu app (botón *«JSON para render»*) en un **MP4 terminado**:
imagen por escena → movimiento (Ken Burns) → locución (voz IA) → música → texto en pantalla.

**Fuentes de imagen (elige en el CONFIG):**
- `drive` — tu **base de datos** de fotos/clips reales en Google Drive (lo más on-brand).
- `stock` — imágenes reales de **internet** vía Pexels (API gratis).
- `ai` — imágenes **generadas por IA** por escena (Pollinations, gratis, sin key).
- `auto` — intenta drive → stock → ai.

Ejecuta las celdas **en orden** (▶). La primera vez, Colab instala dependencias (~1 min).


## 1 · Instalar dependencias

In [ ]:
!pip -q install moviepy==1.0.3 edge-tts nest_asyncio requests
print("✔ Dependencias instaladas.")

## 2 · Configuración
Ajusta la fuente de imágenes y (si aplica) tu API key de Pexels o tu carpeta de Drive.

In [ ]:
import os, json, requests, urllib.parse, asyncio, numpy as np
from PIL import Image, ImageDraw, ImageFont
if not hasattr(Image, "ANTIALIAS"):        # compatibilidad Pillow>=10 con moviepy 1.0.3
    Image.ANTIALIAS = Image.LANCZOS
import nest_asyncio; nest_asyncio.apply()
import edge_tts
from moviepy.editor import (ImageClip, AudioFileClip, CompositeVideoClip,
                            CompositeAudioClip, concatenate_videoclips)

# ======================= CONFIG =======================
SOURCE     = "ai"        # "drive" | "stock" | "ai" | "auto"
PEXELS_KEY = ""          # API key gratis de https://www.pexels.com/api/  (solo para stock)
ASSET_DIR  = "/content/drive/MyDrive/altomonte_assets"   # tu base de datos (SOURCE=drive)
VOICE_ES   = "es-CO-SalomeNeural"   # es-ES-AlvaroNeural · es-MX-JorgeNeural · es-CO-GonzaloNeural
VOICE_EN   = "en-US-AriaNeural"
MUSIC_PATH = ""          # sube un .mp3 y pon su ruta (o deja vacío)
FPS        = 30
OUT        = "alto_monte.mp4"
# ======================================================
WORK = "/content/frames"; os.makedirs(WORK, exist_ok=True)
print("✔ Config lista · fuente =", SOURCE)

### 2b · (Opcional) Montar Google Drive
Solo si `SOURCE = "drive"` o `"auto"` y quieres usar tu propia base de datos de imágenes/clips.

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# print(os.listdir(ASSET_DIR))

## 3 · Pega aquí el JSON de tu app
Reemplaza el contenido entre comillas por el JSON que descargaste (o súbelo y cárgalo). Viene un ejemplo listo.

In [ ]:
SCENE_JSON = r'''
{
  "brand": "ALTO MONTE",
  "tagline": "Energía que nace de las alturas",
  "type": "lanzamiento",
  "style": "cinematografico",
  "aspect": "9:16",
  "resolution": {
    "w": 1080,
    "h": 1920
  },
  "platform": "Instagram Reels / Stories",
  "duration_target": 60,
  "language": "es",
  "voice": true,
  "music": true,
  "onscreen": true,
  "palette": {
    "primary": "#3FE0A0",
    "secondary": "#F4B44C",
    "base": "#080B0A"
  },
  "scenes": [
    {
      "n": 1,
      "beat": "hook",
      "label": "Apertura",
      "seconds": 5,
      "shot": "Aéreo",
      "stock_query": "aerial sunrise mountain range",
      "ai_prompt": "rising crane — first sunrise light cresting a snow-capped mountain range, light sweeping down the slope. Lighting: golden hour, warm backlight. cinematic commercial film, brand color grade of energy green #3FE0A0 and sunrise amber #F4B44C, deep contrast, volumetric light, 4K, filmic, professional",
      "vo": "¿De dónde viene la energía que mueve tu mundo?",
      "onscreen": "",
      "motion": "zoom-out"
    },
    {
      "n": 2,
      "beat": "contexto",
      "label": "Contexto",
      "seconds": 6,
      "shot": "Plano general",
      "stock_query": "city skyline lights dusk aerial",
      "ai_prompt": "gentle orbit — a city waking up, lights switching on street by street at dusk. Lighting: soft diffused natural light. cinematic commercial film, energy green and sunrise amber grade, 4K, filmic",
      "vo": "Cada hogar, cada ciudad, necesita una fuerza que no se agote.",
      "onscreen": "El mundo funciona con energía",
      "motion": "zoom-out"
    },
    {
      "n": 3,
      "beat": "marca",
      "label": "Revelación de marca",
      "seconds": 7,
      "shot": "Plano general",
      "stock_query": "hydroelectric dam water aerial",
      "ai_prompt": "fluid steadicam — a hydroelectric dam releasing water, mist catching the golden light. Lighting: high contrast with deep shadows. cinematic, energy green and amber grade, 4K",
      "vo": "Nace ALTO MONTE. Energía que baja desde las alturas.",
      "onscreen": "ALTO MONTE",
      "motion": "zoom-out"
    },
    {
      "n": 4,
      "beat": "diferencial",
      "label": "Diferencial",
      "seconds": 6,
      "shot": "Plano medio",
      "stock_query": "engineer tablet wind turbine",
      "ai_prompt": "slow dolly-in — an engineer with a tablet in front of a wind turbine, reading live data. Lighting: cool dawn blue with amber accents. cinematic, brand grade, 4K",
      "vo": "Energía renovable e inteligente, gestionada en tiempo real.",
      "onscreen": "Energía inteligente",
      "motion": "zoom-in"
    },
    {
      "n": 5,
      "beat": "demostracion",
      "label": "Demostración",
      "seconds": 6,
      "shot": "Detalle / Primer plano",
      "stock_query": "wind turbine blades closeup sun",
      "ai_prompt": "revealing tilt-up — turbine blades spinning in close-up, sun flares breaking between them. Lighting: volumetric light through haze. cinematic, brand grade, 4K",
      "vo": "De la montaña a tu hogar, sin apagar el planeta.",
      "onscreen": "En acción",
      "motion": "zoom-in"
    },
    {
      "n": 6,
      "beat": "impacto",
      "label": "Impacto",
      "seconds": 7,
      "shot": "Plano general",
      "stock_query": "village lights valley night",
      "ai_prompt": "rising crane — aerial of a connected community, warm lights dotting the valley. Lighting: golden hour, warm backlight. cinematic, brand grade, 4K",
      "vo": "Iluminamos comunidades enteras con energía limpia.",
      "onscreen": "Para las comunidades",
      "motion": "zoom-out"
    },
    {
      "n": 7,
      "beat": "diferencial",
      "label": "Diferencial",
      "seconds": 6,
      "shot": "Plano medio",
      "stock_query": "energy control room screens green",
      "ai_prompt": "gentle orbit — a modern control room, screens showing energy maps and green flows. Lighting: high contrast with deep shadows. cinematic, brand grade, 4K",
      "vo": "",
      "onscreen": "Gestión en tiempo real",
      "motion": "zoom-in"
    },
    {
      "n": 8,
      "beat": "demostracion",
      "label": "Demostración",
      "seconds": 5,
      "shot": "Detalle / Primer plano",
      "stock_query": "water penstock hydro power",
      "ai_prompt": "fluid steadicam — water rushing down the penstock pipe, pure kinetic energy. Lighting: soft diffused natural light. cinematic, brand grade, 4K",
      "vo": "",
      "onscreen": "",
      "motion": "zoom-in"
    },
    {
      "n": 9,
      "beat": "cierre",
      "label": "Cierre / CTA",
      "seconds": 6,
      "shot": "Logo / Cierre",
      "stock_query": "mountain summit sunrise cinematic aerial",
      "ai_prompt": "slow dolly-in — the tagline appears over a majestic aerial of the mountain range at dawn. Lighting: golden hour, warm backlight. cinematic, brand grade, 4K",
      "vo": "ALTO MONTE. Energía que nace de las alturas.",
      "onscreen": "Energía que nace de las alturas",
      "motion": "zoom-out"
    }
  ]
}
'''

data = json.loads(SCENE_JSON)
W, H = data['resolution']['w'], data['resolution']['h']
LANG  = data.get('language','es')
VOICE = VOICE_ES if LANG=='es' else VOICE_EN
print(f"{data['brand']} · {len(data['scenes'])} escenas · {W}x{H} · idioma={LANG}")

## 4 · Funciones (imágenes, voz, texto, movimiento)

In [ ]:
def hex2rgb(h):
    h = h.lstrip("#"); return tuple(int(h[i:i+2],16) for i in (0,2,4))

def cover(img, w, h):
    """Escala para cubrir wxh y recorta al centro (sin deformar)."""
    iw, ih = img.size; s = max(w/iw, h/ih)
    img = img.resize((int(iw*s)+1, int(ih*s)+1), Image.LANCZOS)
    iw, ih = img.size; x=(iw-w)//2; y=(ih-h)//2
    return img.crop((x, y, x+w, y+h))

def _save(content, path):
    with open(path, "wb") as f: f.write(content)
    return path

def get_stock(query, path):
    r = requests.get("https://api.pexels.com/v1/search",
        headers={"Authorization": PEXELS_KEY},
        params={"query": query, "per_page": 1,
                "orientation": "portrait" if H>W else ("square" if H==W else "landscape")},
        timeout=30)
    photos = r.json().get("photos", [])
    if not photos: raise RuntimeError("stock sin resultados: " + query)
    return _save(requests.get(photos[0]["src"]["large2x"], timeout=60).content, path)

def get_ai(prompt, path):
    p = urllib.parse.quote(prompt[:380])
    url = f"https://image.pollinations.ai/prompt/{p}?width={W}&height={H}&nologo=true&seed=7"
    return _save(requests.get(url, timeout=120).content, path)

def list_drive():
    if not os.path.isdir(ASSET_DIR): return []
    ex = (".jpg",".jpeg",".png",".webp")
    return sorted(os.path.join(ASSET_DIR,f) for f in os.listdir(ASSET_DIR) if f.lower().endswith(ex))

DRIVE = list_drive() if SOURCE in ("drive","auto") else []

def get_image(sc, i, path):
    order = [SOURCE] if SOURCE != "auto" else ["drive","stock","ai"]
    for src in order:
        try:
            if src=="drive" and DRIVE:
                return _save(open(DRIVE[i % len(DRIVE)], "rb").read(), path)
            if src=="stock" and PEXELS_KEY:
                return get_stock(sc["stock_query"], path)
            if src=="ai":
                return get_ai(sc["ai_prompt"], path)
        except Exception as e:
            print("   ⚠", src, "→", e)
    return get_ai(sc["ai_prompt"], path)   # último recurso

# ---- voz en off (edge-tts, gratis) ----
async def _tts(text, path):
    await edge_tts.Communicate(text, VOICE).save(path)
def tts(text, path):
    asyncio.get_event_loop().run_until_complete(_tts(text, path)); return path

# ---- texto en pantalla (PIL, sin ImageMagick) ----
FONT = "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf"
def _wrap(draw, text, font, maxw):
    words = text.split(); lines=[]; cur=""
    for w in words:
        t = (cur+" "+w).strip()
        if draw.textlength(t, font=font) <= maxw: cur = t
        else: lines.append(cur); cur = w
    if cur: lines.append(cur)
    return lines
def text_png(text, rgb):
    im = Image.new("RGBA", (W,H), (0,0,0,0)); d = ImageDraw.Draw(im)
    fs = int(W*0.052); font = ImageFont.truetype(FONT, fs)
    lines = _wrap(d, text.upper(), font, int(W*0.86)); lh = int(fs*1.28)
    total = lh*len(lines); y = int(H*0.80) - total
    d.rectangle([0, y-int(fs*0.6), W, y+total+int(fs*0.4)], fill=(8,11,10,140))
    for ln in lines:
        x = (W - d.textlength(ln, font=font))//2
        d.text((x, y), ln, font=font, fill=rgb); y += lh
    p = os.path.join(WORK, "txt.png"); im.save(p); return p

# ---- movimiento Ken Burns ----
def ken_burns(path, dur, zin=True, zoom=0.10):
    base = cover(Image.open(path).convert("RGB"), W, H)
    p = path + "_c.jpg"; base.save(p, quality=92)
    clip = ImageClip(p).set_duration(dur).set_position(("center","center"))
    clip = clip.resize((lambda t: 1 + zoom*(t/dur)) if zin
                       else (lambda t: (1+zoom) - zoom*(t/dur)))
    return CompositeVideoClip([clip], size=(W,H)).set_duration(dur)

print("✔ Funciones cargadas.")

## 5 · Construir cada escena

In [ ]:
prim = hex2rgb(data["palette"]["primary"])
clips = []
for i, sc in enumerate(data["scenes"]):
    print(f'Escena {sc["n"]}/{len(data["scenes"])} · {sc["label"]}')
    img = get_image(sc, i, os.path.join(WORK, f"img{i}.jpg"))

    dur = float(sc["seconds"]); audio = None
    if data.get("voice") and sc.get("vo"):
        vp = os.path.join(WORK, f"vo{i}.mp3"); tts(sc["vo"], vp)
        audio = AudioFileClip(vp); dur = max(dur, audio.duration + 0.5)

    layers = [ken_burns(img, dur, zin=(sc.get("motion") != "zoom-out"))]
    if data.get("onscreen") and sc.get("onscreen"):
        layers.append(ImageClip(text_png(sc["onscreen"], prim))
                      .set_duration(dur).set_position(("center","center")))

    scene = CompositeVideoClip(layers, size=(W,H)).set_duration(dur)
    if audio is not None: scene = scene.set_audio(audio)
    clips.append(scene)

video = concatenate_videoclips(clips, method="compose")
print("✔ Duración total:", round(video.duration,1), "s")

## 6 · Música + exportar MP4

In [ ]:
if MUSIC_PATH and os.path.exists(MUSIC_PATH):
    from moviepy.audio.fx.all import audio_loop
    m = AudioFileClip(MUSIC_PATH).volumex(0.12)
    m = audio_loop(m, duration=video.duration) if m.duration < video.duration else m.subclip(0, video.duration)
    base = video.audio
    video = video.set_audio(CompositeAudioClip([base, m]) if base else m)

video.write_videofile(OUT, fps=FPS, codec="libx264", audio_codec="aac",
                      threads=4, preset="medium")
print("✅ Listo:", OUT)
from google.colab import files; files.download(OUT)

## 7 · (Opcional) Video generativo real por escena (API de pago)

Si quieres que **una toma estrella** sea video IA de verdad (no imagen animada), sustituye esa
escena por un clip de Runway / Luma / Kling vía [Replicate](https://replicate.com). Requiere token y tiene costo.
El resultado se une igual que las demás escenas.


In [ ]:
# !pip -q install replicate
# import replicate, os
# os.environ["REPLICATE_API_TOKEN"] = "r8_xxx"   # tu token
# out = replicate.run(
#     "MODELO/VERSION",                            # p.ej. un modelo image/text-to-video
#     input={"prompt": data["scenes"][0]["ai_prompt"], "aspect_ratio": data["aspect"]}
# )
# print(out)   # URL del clip .mp4 → descárgalo y móntalo en la lista de clips

## Notas

- **Tu base de datos:** pon tus fotos/clips en `ASSET_DIR` (Drive), `SOURCE="drive"`. Se usan en orden
  de escena; si faltan, con `SOURCE="auto"` completa con stock o IA.
- **Stock gratis:** saca tu key en pexels.com/api y pégala en `PEXELS_KEY`.
- **Voces:** cambia `VOICE_ES` (ej. `es-ES-AlvaroNeural`, `es-MX-JorgeNeural`).
- **Formato:** el JSON ya trae la resolución (9:16, 1:1, 16:9, 4:5) según lo que elegiste en la app.
- **Filosofía tipo Oreate:** este pipeline combina texto (locución) + imágenes + audio → video,
  igual que un workspace todo-en-uno, pero corriendo en tu Colab y con tu material.
